# Customer Churn Prediction System
### *"Predict. Prevent. Retain."*

**Course Project:** Intermediate Machine Learning Mini-Project  
**System Name:** ChurnGuard AI  
**Dataset:** IBM Telco Customer Churn (7,043 records, 21 attributes)  
**Target:** `Churn` (0 = Retained, 1 = Churned)  

---

## 1. Project Workflow Outline
1. **Problem Definition** - Identify high-risk subscribers to reduce revenue defection.
2. **Dataset Understanding** - Multi-attribute inspection of demographic, service, and billing signals.
3. **Data Preprocessing & Cleaning** - Blank value coercion, type casting, no data leakage.
4. **Exploratory Data Analysis (EDA)** - Univariate, bivariate, and cross-tabulation.
5. **Feature Engineering & Transformation** - StandardScaler + OneHotEncoder pipeline.
6. **Feature Selection** - Tri-method analysis: Mutual Information, ANOVA F-test, RF Gini.
7. **Model Development** - Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost.
8. **Hyperparameter Tuning** - 5-Fold Stratified GridSearchCV optimization.
9. **Model Evaluation & Comparison** - Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix.
10. **Why Performances Differ** - Comparative algorithmic theory.
11. **Model Persistence** - Serializing best model and pipeline with joblib.
12. **Inference & Explainability** - Simulated real-time scoring with retention actions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

# Set display aesthetics
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 2. Dataset Ingestion & Cleaning

In [ ]:
df = pd.read_csv('../data/customer_churn.csv')
print(f"Raw dataset shape: {df.shape}")
df.head()

In [ ]:
# Convert TotalCharges from string to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce').fillna(0.0)

# Check target distribution
churn_dist = df['Churn'].value_counts(normalize=True) * 100
print("Target Distribution (%):\n", churn_dist)

# Drop identifier customerID
df_clean = df.drop(columns=['customerID'])
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})

## 3. Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Contract vs Churn
sns.countplot(data=df, x='Contract', hue='Churn', palette='Set2', ax=axes[0])
axes[0].set_title('Churn by Contract Type')

# 2. Internet Service vs Churn
sns.countplot(data=df, x='InternetService', hue='Churn', palette='Set1', ax=axes[1])
axes[1].set_title('Churn by Internet Service')

# 3. Tenure vs Churn Boxplot
sns.boxplot(data=df, x='Churn', y='tenure', palette='coolwarm', ax=axes[2])
axes[2].set_title('Tenure Distribution by Churn Status')

plt.tight_layout()
plt.show()

## 4. Preprocessing Pipeline & Stratified Train-Test Split

In [ ]:
X = df_clean.drop(columns=['Churn'])
y = df_clean['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [c for c in X.columns if c not in num_cols]

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols)
])

X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)

ohe_names = preprocessor.named_transformers_['cat'].get_feature_names_out(cat_cols).tolist()
feature_names = num_cols + ohe_names

X_train_df = pd.DataFrame(X_train_trans, columns=feature_names)
X_test_df = pd.DataFrame(X_test_trans, columns=feature_names)
print(f"Transformed Training Matrix Shape: {X_train_df.shape}")

## 5. Feature Selection & Importance Ranking

In [ ]:
# Mutual Information
mi_scores = mutual_info_classif(X_train_df, y_train, random_state=42)
# Random Forest Feature Importance
rf_fs = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_fs.fit(X_train_df, y_train)

feat_imp_df = pd.DataFrame({
    'Feature': feature_names,
    'Mutual_Info': mi_scores,
    'RF_Importance': rf_fs.feature_importances_
}).sort_values(by='Mutual_Info', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_imp_df.head(10), x='Mutual_Info', y='Feature', palette='viridis')
plt.title('Top 10 Features Ranked by Mutual Information')
plt.show()

## 6. Baseline Model Training

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss')
}

results = []
for name, model in models.items():
    model.fit(X_train_df, y_train)
    preds = model.predict(X_test_df)
    probs = model.predict_proba(X_test_df)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds), 4),
        'Recall': round(recall_score(y_test, preds), 4),
        'F1 Score': round(f1_score(y_test, preds), 4),
        'ROC-AUC': round(roc_auc_score(y_test, probs), 4)
    })

pd.DataFrame(results)

## 7. Hyperparameter Tuning with GridSearchCV

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.03, 0.1],
    'subsample': [0.8, 1.0]
}

xgb_cv = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid=xgb_param_grid,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)
xgb_cv.fit(X_train_df, y_train)
best_xgb = xgb_cv.best_estimator_
print("Best XGBoost Parameters:", xgb_cv.best_params_)

tuned_preds = best_xgb.predict(X_test_df)
tuned_probs = best_xgb.predict_proba(X_test_df)[:, 1]
print(f"Tuned XGBoost Holdout ROC-AUC: {roc_auc_score(y_test, tuned_probs):.4f}")

## 8. Confusion Matrix & ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, tuned_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
            xticklabels=['Retained (0)', 'Churned (1)'],
            yticklabels=['Retained (0)', 'Churned (1)'])
plt.title('Tuned XGBoost Confusion Matrix (Holdout Test Set)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

## 9. Simulated Real-Time Prediction
Simulating customer scoring with risk-tier classification.

In [ ]:
sample_customer = X.iloc[[0]]
sample_transformed = preprocessor.transform(sample_customer)
prob = best_xgb.predict_proba(sample_transformed)[0, 1]

risk_tier = "HIGH RISK" if prob > 0.65 else ("MEDIUM RISK" if prob >= 0.35 else "LOW RISK")
print(f"Simulated Subscriber Churn Probability: {prob * 100:.1f}%")
print(f"Assigned Risk Tier: {risk_tier}")